# 04 — Generate ROI Dataset

Runs the contour-based stamp detection pipeline across the entire dataset and saves cropped, standardised 224×224 px stamp ROIs to disk as the training input for the classifier in notebook 05.

> This notebook is a batch processing script — run it once to generate the ROI dataset, then proceed to notebook 05 for training.

**Input:** `final_dataset/class_0_genuine/` and `final_dataset/class_1_forged/`  
**Output:** `outputs/roi_dataset_v3/genuine/` and `outputs/roi_dataset_v3/forged/`

---
## Contents
1. Imports
2. Configuration — paths and ROI size
3. Helper: `show_image`
4. Function: `contour_stamp_score`
5. Function: `find_stamp_candidates_by_contours`
6. Function: `extract_roi_from_candidate_original` — scale back to full resolution
7. Function: `standardize_roi` — resize to 224×224
8. Function: `save_roi_image`
9. Discover all images
10. Batch ROI extraction loop
11. Extraction summary

## 1. Imports

In [1]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

## 2. Configuration — paths and ROI size

Sets up all paths and constants needed before the batch processing loop runs. The output folders are created here at the top so they are ready to receive the extracted ROIs before any processing begins.

| Variable | Value | Description |
|---|---|---|
| `DATASET_ROOT` | local path | Raw scanned document images |
| `OUTPUT_ROOT` | `../outputs/roi_dataset_v3/` | Destination for extracted ROIs |
| `GENUINE_OUTPUT` | `OUTPUT_ROOT/genuine/` | ROIs for class 0 |
| `FORGED_OUTPUT` | `OUTPUT_ROOT/forged/` | ROIs for class 1 |
| `ROI_SIZE` | 224 | Output image size in px — matches the input size expected by ResNet50 |

> ⚠️ Update `DATASET_ROOT` to your local path before running.

In [2]:
DATASET_ROOT = Path(
    r"D:\sem-07\EC-7205-computer-vision-image-processing\CA\project\final_dataset"
)

OUTPUT_ROOT = Path("../outputs/roi_dataset_v3")

GENUINE_OUTPUT = OUTPUT_ROOT / "genuine"
FORGED_OUTPUT = OUTPUT_ROOT / "forged"

GENUINE_OUTPUT.mkdir(parents=True, exist_ok=True)
FORGED_OUTPUT.mkdir(parents=True, exist_ok=True)

ROI_SIZE = 224

## 3. Helper: `show_image`

In [3]:
def show_image(title, image, cmap=None, figsize=(8, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 4. Function — `contour_stamp_score`

Scores a single contour based on how circular and compact it is. Contours with area < 200 px² or compactness < 0.35 are rejected and returned as `None`. The remaining contours are scored as a weighted combination of compactness (45%), circularity (35%), and extent (20%).

Duplicated from notebook 03 so this notebook can run independently without importing from previous notebooks.

In [4]:
def contour_stamp_score(contour):
    area = cv2.contourArea(contour)

    if area <= 0:
        return None

    x, y, w, h = cv2.boundingRect(contour)
    bbox_area = w * h

    if bbox_area <= 0:
        return None

    aspect_ratio = w / h
    compactness = min(w, h) / max(w, h)
    extent = area / bbox_area

    perimeter = cv2.arcLength(contour, True)

    if perimeter <= 0:
        circularity = 0
    else:
        circularity = (4 * np.pi * area) / (perimeter ** 2)

    # Basic rejection rules
    if area < 200:
        return None

    if compactness < 0.35:
        return None

    # Weighted score
    score = (
        0.45 * compactness +
        0.35 * circularity +
        0.20 * extent
    )

    return {
        "contour": contour,
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "area": area,
        "aspect_ratio": aspect_ratio,
        "compactness": compactness,
        "extent": extent,
        "circularity": circularity,
        "score": score
    }

## 5. Function — `find_stamp_candidates_by_contours`

Runs the full stamp detection pipeline on a single image in one call — resizing, three-channel HSV thresholding, morphological opening (3×3) then closing (11×11), contour detection, scoring, and ranking. Returns the candidates sorted by score highest first.

**Returns:** `(image_resized, mask_cleaned, candidates)`

In [5]:
def find_stamp_candidates_by_contours(image_bgr, display_width=900):
    h0, w0 = image_bgr.shape[:2]

    scale = display_width / w0
    new_h = int(h0 * scale)

    image_resized = cv2.resize(image_bgr, (display_width, new_h))

    image_hsv = cv2.cvtColor(image_resized, cv2.COLOR_BGR2HSV)

    h = image_hsv[:, :, 0]
    s = image_hsv[:, :, 1]
    v = image_hsv[:, :, 2]

    hue_mask = cv2.inRange(h, 90, 170)
    sat_mask = cv2.inRange(s, 25, 255)
    val_mask = cv2.inRange(v, 30, 255)

    mask = cv2.bitwise_and(hue_mask, sat_mask)
    mask = cv2.bitwise_and(mask, val_mask)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

    mask_opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask_cleaned = cv2.morphologyEx(mask_opened, cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(
        mask_cleaned,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for contour in contours:
        result = contour_stamp_score(contour)

        if result is not None:
            candidates.append(result)

    candidates = sorted(
        candidates,
        key=lambda item: item["score"],
        reverse=True
    )

    return image_resized, mask_cleaned, candidates

## 6. Function — `extract_roi_from_candidate_original`

The detection pipeline runs on the resized 900px image — so the candidate bounding box coordinates are in resized image coordinates, not original image coordinates. Cropping directly using those coordinates on the original image would crop the wrong region.

This function converts the resized coordinates back to original full-resolution coordinates by calculating a scale factor between the two image sizes (`scale_x` and `scale_y`) and multiplying the bounding box coordinates by those factors before cropping.

The crop is taken from the original full-resolution image rather than the resized one because the classifier needs to analyse microscopic ink texture differences between genuine and forged stamps — this detail only exists at the scanner's native resolution (300 or 600 DPI) and would be lost if the ROI was extracted from the downscaled 900px image.

**Parameters:**
- `original_image` — full-resolution BGR image
- `resized_image` — 900px wide version used for detection
- `candidate` — dict from `find_stamp_candidates_by_contours`
- `pad_factor` — padding as fraction of bounding box size (default 1.0)

In [6]:
def extract_roi_from_candidate_original(
    original_image,
    resized_image,
    candidate,
    display_width=900,
    pad_factor=1.0
):
    original_h, original_w = original_image.shape[:2]
    resized_h, resized_w = resized_image.shape[:2]

    scale_x = original_w / resized_w
    scale_y = original_h / resized_h

    x, y, w, h = candidate["x"], candidate["y"], candidate["w"], candidate["h"]

    pad_x = int(w * pad_factor)
    pad_y = int(h * pad_factor)

    # bbox in resized coordinates
    x1 = max(x - pad_x, 0)
    y1 = max(y - pad_y, 0)
    x2 = min(x + w + pad_x, resized_w)
    y2 = min(y + h + pad_y, resized_h)

    # convert to original coordinates
    ox1 = int(x1 * scale_x)
    oy1 = int(y1 * scale_y)
    ox2 = int(x2 * scale_x)
    oy2 = int(y2 * scale_y)

    roi_original = original_image[oy1:oy2, ox1:ox2]

    return roi_original, (ox1, oy1, ox2, oy2)

## 7. Function — `standardize_roi`

Resizes any ROI crop to a fixed 224×224 px square before saving. All images fed to the ResNet50 classifier must be exactly this size.

`INTER_AREA` interpolation is used because it averages pixel values in the area being downscaled — producing a cleaner result than other interpolation methods and preserving as much microscopic texture detail as possible during resizing.

In [7]:
def standardize_roi(roi_bgr, size=224):

    roi_resized = cv2.resize(
        roi_bgr,
        (size, size),
        interpolation=cv2.INTER_AREA
    )

    return roi_resized

## 8. Function: `save_roi_image`

Saves a BGR ROI array to disk as PNG using `cv2.imwrite`. Returns `True` on success.


In [8]:
def save_roi_image(roi_bgr, output_path):

    success = cv2.imwrite(
        str(output_path),
        roi_bgr
    )

    return success

## 9. Discover All Images

Recursively finds all `.png` files under `DATASET_ROOT`. This should return all genuine and forged samples (~300 total).


In [9]:
all_images = list(DATASET_ROOT.rglob("*.png"))

print("Total images:", len(all_images))

Total images: 540


## 10. Batch ROI Extraction Loop

Iterates over all images in the dataset with a `tqdm` progress bar. For each image it attempts to detect and extract the stamp ROI and records the outcome in `records`.

There are four possible failure points, each logged separately:
- **`image_read_failed`** — `cv2.imread` returned `None`, the file is corrupt or unreadable
- **`no_candidate_found`** — the pipeline found no contours that passed the scoring threshold
- **`no_candidate_selected`** — `candidates[0]` returned `None`
- **`empty_roi`** — the crop produced an empty array due to invalid bounding box coordinates

If all steps succeed, the ROI is saved to `GENUINE_OUTPUT` or `FORGED_OUTPUT` depending on which class folder the source image came from. All outcomes are recorded in `records` and converted to `df_roi` at the end for the extraction summary.

In [10]:
records = []

for image_path in tqdm(all_images, desc="Extracting ROIs"):
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "image_read_failed"
        })
        continue

    image_resized, mask_cleaned, candidates = find_stamp_candidates_by_contours(image_bgr)

    if len(candidates) == 0:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "no_candidate_found"
        })
        continue

    best_candidate = candidates[0]

    if best_candidate is None:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "no_candidate_selected"
        })
        continue

    roi, bbox = extract_roi_from_candidate_original(
        original_image=image_bgr,
        resized_image=image_resized,
        candidate=best_candidate,
        display_width=900,
        pad_factor=1.0
    )

    if roi is None or roi.size == 0:
        records.append({
            "source_path": str(image_path),
            "roi_saved": False,
            "reason": "empty_roi"
        })
        continue

    roi_standardized = roi  # save high-resolution ROI, not 224x224 yet

    class_name = "genuine" if "class_0_genuine" in str(image_path) else "forged"
    output_dir = GENUINE_OUTPUT if class_name == "genuine" else FORGED_OUTPUT

    output_name = image_path.stem + "_roi.png"
    output_path = output_dir / output_name

    success = save_roi_image(roi_standardized, output_path)

    records.append({
        "source_path": str(image_path),
        "roi_path": str(output_path),
        "class_name": class_name,
        "roi_saved": success,
        "reason": "success" if success else "save_failed",
        "candidate_score": best_candidate["score"],
        "candidate_x": best_candidate["x"],
        "candidate_y": best_candidate["y"],
        "candidate_w": best_candidate["w"],
        "candidate_h": best_candidate["h"],
        "bbox": bbox,
        "num_candidates": len(candidates),
        "selected_candidate_rank": 0,
        "ink_score": float(best_candidate["circularity"]),
        "shape_score": float(best_candidate["compactness"]),
        "final_score": float(best_candidate["score"]),
    })

df_roi = pd.DataFrame(records)

df_roi.head()

Extracting ROIs: 100%|██████████| 540/540 [02:23<00:00,  3.75it/s]


,source_path,roi_path,class_name,roi_saved,reason,candidate_score,candidate_x,candidate_y,candidate_w,candidate_h,bbox,num_candidates,selected_candidate_rank,ink_score,shape_score,final_score
0,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.764262,644.0,1083.0,70.0,81.0,"(1582, 2763, 2161, 3434)",1.0,0.0,0.671940,0.864198,0.764262
1,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.841667,173.0,1004.0,80.0,83.0,"(256, 2540, 917, 3227)",1.0,0.0,0.747146,0.963855,0.841667
2,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.836470,634.0,797.0,83.0,83.0,"(1518, 1969, 2205, 2656)",1.0,0.0,0.706215,1.000000,0.836470
3,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.888471,676.0,722.0,84.0,85.0,"(1631, 1757, 2326, 2460)",1.0,0.0,0.846333,0.988235,0.888471
4,D:\sem-07\EC-7205-computer-vision-image-proces...,..\outputs\roi_dataset_v3\forged\Forged_Laser_...,forged,True,success,0.505128,645.0,1086.0,69.0,78.0,"(3175, 5557, 4316, 6847)",1.0,0.0,0.113314,0.884615,0.505128


## 11. Extraction Summary

Prints two summaries — overall extraction success and failure counts, and the same counts broken down by class.

Review these before proceeding to notebook 05:
- A high failure count overall means the pipeline is not detecting stamps reliably and the parameters in `contour_stamp_score` may need adjustment
- A large difference in failure counts between genuine and forged classes means the training dataset will be imbalanced, which could bias the classifier toward the majority class

In [11]:
print("ROI extraction summary:")
print(df_roi["roi_saved"].value_counts())

print("\nClass-wise summary:")
print(df_roi.groupby(["class_name", "roi_saved"]).size())

ROI extraction summary:
roi_saved
True     518
False     22
Name: count, dtype: int64

Class-wise summary:
class_name  roi_saved
forged      True         205
genuine     True         313
dtype: int64
